

# Prompt Tool Delta Embedding and Cosine Baseline ToolBench Mismatch Ratio Study Experiment

This notebook runs a class-ratio sensitivity study for the delta-embedding detector and a cosine-similarity baseline.

Base data: `dataset/toolbench_mismatch_dataset.csv`

Experiment design:

- Keep label 0 normal samples fixed at 100,000 rows.
- Reduce label 1 mismatch samples according to Normal:Mismatch ratios: 1:1, 7:3, 8:2, 10:1, 100:1.
- Use the same delta feature: `prompt_embedding - tool_embedding`.
- Compare against a direct cosine-similarity baseline on the 1:1 ratio using the same prompt/tool embeddings.
- Use the same model set: Cosine Similarity, LightGBM, XGBoost, Random Forest, Isolation Forest.
- Use `source_prompt_hash` as the group split key to avoid prompt-level leakage.
- Report Precision, Recall, F1-score, Accuracy, AUROC, AUPRC, confusion matrix, and latency.

Note: the 7:3 case uses 42,857 label 1 rows because 100,000 * 3 / 7 is not an integer.


## Cell 1. Install packages


In [ ]:
%pip install pandas numpy scikit-learn matplotlib lightgbm sentence-transformers torch xgboost 
# 노트북 실행에 필요한 패키지들을 현재 python 환경에 설치.


## Cell 2. Imports and experiment settings


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)


DATASET_RELATIVE_PATH = Path('dataset') / 'toolbench_mismatch_dataset.csv' #데이터셋 경로 저장


def find_repository_root() -> Path: #최상위 폴더를 찾아주는 함수
    cwd = Path.cwd().resolve() #현재 작업 디렉토리를 계산
    for candidate in [cwd, *cwd.parents]: #cwd와 그 부모 디렉토리들을 반복 처리
        if (candidate / 'notebooks').exists() and (candidate / 'dataset').exists(): #notebooks와 dataset 폴더가 존재하는지 확인
            return candidate #존재하면 해당 디렉토리를 반환
    return cwd.parent if cwd.name == 'notebooks' else cwd #notebooks 폴더가 없으면 현재 디렉토리의 부모 디렉토리를 반환


def resolve_dataset_path(repo_root: Path) -> Path: #프로젝트 루트 경로를 기준으로 실제 데이터 셋 파일 경로를 만들고 그파일 존재하는지 확인하는 함수
    data_path = repo_root / DATASET_RELATIVE_PATH #데이터셋 경로를 계산
    if data_path.exists(): #데이터셋 파일이 존재하는지 확인
        return data_path #존재하면 해당 경로를 반환
    raise FileNotFoundError(  #예외 처리
        'Could not find the dataset file.\n'
        f'Expected relative path: {DATASET_RELATIVE_PATH}\n'
        f'Resolved path: {data_path}'
    )


ROOT = find_repository_root() #프로젝트 루트 경로 계산
DATA_PATH = resolve_dataset_path(ROOT) #데이터셋 경로 계산

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S') #실험 결과 폴더에 이름 붙여서 타임스탬프로 구분가능하게 함
RUN_ROOT = ROOT / 'outputs' / 'runs' / f'toolbench_mismatch_ratio_study_cpu_models_{RUN_TIMESTAMP}' # 실험 결과를 저장할 폴더 경로
RUN_ROOT.mkdir(parents=True, exist_ok=True) #출력 폴더 경로 생성

#실험 세팅
TRAIN_RATIO = 0.64 #전체 데이터중 64%를 학습 데이터로 사용
VALID_RATIO = 0.16 #전체 데이터중 16%를 검증 데이터로 사용
TEST_RATIO = 0.20 #전체 데이터중 20%를 테스트 데이터로 사용
SPLIT_GROUP_COL = 'source_prompt_hash' #프롬프트 해시 열을 그룹 분할 키로 사용
RANDOM_SEED = 42 #랜덤시드 42로 고정
DEFAULT_THRESHOLDS = np.round(np.arange(0.0, 1.01, 0.01), 2) #0.01 간격으로 threshold 후보 생성
LATENCY_SAMPLE_SIZE = 100 #추론 시간 측정에 사용할 샘플 개수
RUN_LATENCY_BENCHMARK = True #추론시간 밴치마크 실행 여부

NORMAL_SAMPLE_SIZE = 100_000 #정상 샘플 10만개 사용
RATIO_CONFIGS = [ # 실험에서 사용할 비율 설정
    {'ratio_name': '1_1', 'ratio_label': '1:1', 'normal_parts': 1, 'negative_parts': 1},
    {'ratio_name': '7_3', 'ratio_label': '7:3', 'normal_parts': 7, 'negative_parts': 3},
    {'ratio_name': '8_2', 'ratio_label': '8:2', 'normal_parts': 8, 'negative_parts': 2},
    {'ratio_name': '10_1', 'ratio_label': '10:1', 'normal_parts': 10, 'negative_parts': 1},
    {'ratio_name': '100_1', 'ratio_label': '100:1', 'normal_parts': 100, 'negative_parts': 1},
]
MAX_NEGATIVE_SAMPLE_SIZE = max( #각 비율 실험에서 필요한 비정상 샘플 개수 계산하고 가잗 큰값 저장
    int(round(NORMAL_SAMPLE_SIZE * cfg['negative_parts'] / cfg['normal_parts']))
    for cfg in RATIO_CONFIGS
)

ANALYSIS_SAMPLE_PER_LABEL = 10_000 #시각화 pca,L2 norm 히스토그램을 그릴때 사용하는 샘플
SILHOUETTE_SAMPLE_PER_LABEL = 1_000 #시각화 silhouette score 계산에 사용하는 샘플

EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2' #사용할 임베딩 모델 이름
EMBEDDING_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' #GPU 사용이 가능하면 'cuda'를 사용, 불가능한경우 cpu 사용
EMBEDDING_BATCH_SIZE = 256 if EMBEDDING_DEVICE == 'cuda' else 64 #임베딩 모델 입력 배치 크기 설정

PREFER_GPU_LIGHTGBM = False #각 모델별 공정성을 위해 모두 cpu를 사용하여 모델 학습에 사용
LIGHTGBM_DEVICE_CANDIDATES = ['cpu']
XGBOOST_DEVICE_CANDIDATES = ['cpu']

MODEL_ORDER = [ #실험 결과 출력을 위한 리스트
    'cosine_similarity',
    'lightgbm',
    'xgboost',
    'random_forest',
    'isolation_forest',
]
COSINE_BASELINE_RATIO_NAMES = {'1_1'} #코사인 유사도는 1:1 비율만 사용


def get_model_order_for_ratio(ratio_name): #ratio_name을 입력받아서 해당 비율 실험에서 사용할 모델 목록을 반환
    if ratio_name in COSINE_BASELINE_RATIO_NAMES: #현재 ratio_name이 코사인 유사도를 포함해야하는 비율인지 확인
        return MODEL_ORDER
    return [model_name for model_name in MODEL_ORDER if model_name != 'cosine_similarity'] #1:1이 아닌 다른 비율에서는 코사인 유사도를 제외한 모델목록 반환

plt.style.use('default') #matplotlib 그래프 스타일 default로 사용

#로그 출력
print('Dataset path         :', DATA_PATH) 
print('Run root             :', RUN_ROOT)
print('Split ratio          :', f'train={TRAIN_RATIO:.2f}, valid={VALID_RATIO:.2f}, test={TEST_RATIO:.2f}')
print('Split group column   :', SPLIT_GROUP_COL)
print('Normal sample size   :', NORMAL_SAMPLE_SIZE)
print('Max negative size    :', MAX_NEGATIVE_SAMPLE_SIZE)
print('Ratio configs        :', [(cfg['ratio_label'], int(round(NORMAL_SAMPLE_SIZE * cfg['negative_parts'] / cfg['normal_parts']))) for cfg in RATIO_CONFIGS])
print('Embedding model      :', EMBEDDING_MODEL_NAME)
print('Embedding device     :', EMBEDDING_DEVICE)
if EMBEDDING_DEVICE == 'cuda':
    print('Embedding GPU name   :', torch.cuda.get_device_name(0))
else:
    print('Embedding GPU name   :', 'CUDA not available -> CPU fallback')
print('Embedding batch size :', EMBEDDING_BATCH_SIZE)
print('LightGBM devices     :', LIGHTGBM_DEVICE_CANDIDATES)
print('XGBoost devices      :', XGBOOST_DEVICE_CANDIDATES)
print('Model order          :', MODEL_ORDER)
print('Cosine baseline ratio:', sorted(COSINE_BASELINE_RATIO_NAMES))
print('Feature design       :', 'delta embedding models plus cosine similarity baseline')


## Cell 3. Shared helper functions


In [ ]:
_EMBEDDING_MODEL = None #임베딩 모델을 저장해둘 전역 변수, 캐시 역할


def normalize_text(value): #텍스트 값을 정리하는 함수
    if pd.isna(value): #값이 없으면 빈 문자열 반환
        return ''
    return ' '.join(str(value).strip().split()) #문자열 양쪽 공백 제거 후 공백으로 구분된 문자열 반환


def get_embedding_model(): #임베딩 모델을 불러오는 함수, 모델을 매번 새로 불러오지 않게 함
    global _EMBEDDING_MODEL #함수 안에서 전역 변수 _EMBEDDING_MODEL을 수정
    if _EMBEDDING_MODEL is None: #임베딩 모델이 없으면 모델 로드
        print(f'Loading embedding model: {EMBEDDING_MODEL_NAME} on {EMBEDDING_DEVICE}') #어떤 임베딩 모델을 어떤 장치에 로드하는지 출력
        _EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME, device=EMBEDDING_DEVICE) #모델 불러오기
    return _EMBEDDING_MODEL


def synchronize_if_needed(): #GPU를 사용할 경우 GPU 작업이 끝날 때까지 기다리는 함수, 시간측정 정확도를 높이기 위해 사용
    if EMBEDDING_DEVICE == 'cuda':
        torch.cuda.synchronize()


def encode_unique_text_column(series, prefix, model): #고유 텍스트 열을 인코딩하는 함수. 같은 텍스트는 한 번만 임베딩. 최적화 전략.
    prepared = series.fillna('').astype(str).map(normalize_text) #빈 문자열로 채워진 경우 빈 문자열로 채움
    codes, uniques = pd.factorize(prepared, sort=False) #인코딩 결과 및 고유 값 반환
    unique_texts = uniques.tolist() #고유 값 리스트로 변환
    show_progress = len(unique_texts) > 1000 #1000개 이상의 고유 값이 있는 경우 진행 상황 표시

    print(f'Encoding {prefix}: unique texts={len(unique_texts):,}')
    unique_embeddings = model.encode( #고유 텍스트들을 임베딩
        unique_texts, #임베딩할 텍스트 목록
        batch_size=EMBEDDING_BATCH_SIZE, #배치 크기
        convert_to_numpy=True, #결과를 NumPy배열로 받음
        normalize_embeddings=True, #임베딩 벡터를 정규화
        show_progress_bar=show_progress, #진행 상황 표시
    )
    unique_embeddings = np.asarray(unique_embeddings, dtype=np.float32) #임베딩 결과를 float32타입으로 변환. 메모리 사용전략
    row_embeddings = unique_embeddings[codes] #고유 텍스트 임베딩을 원래 데이터 행 개수에 맞게 복원

    feature_cols = [f'{prefix}_emb_{index:03d}' for index in range(row_embeddings.shape[1])] #임베딩 벡터 차원 수만큼 컬럼이름 생성
    feature_frame = pd.DataFrame(row_embeddings, columns=feature_cols, index=series.index, dtype=np.float32) #임베딩 배열을 pandas DataFrame으로 바꿈
    return feature_frame, feature_cols


def build_delta_feature_frame(df): #실험의 핵심인 차이벡터 구성하는 함수
    metadata_cols = [
        col
        for col in [ #분석에 참고할 수 잇는 메타데이터 칼럼 목록을 생성.
            'prompt_source_file',
            'prompt_source_toolkit',
            'tool_source_file',
            'tool_source_toolkit',
            'source_prompt_hash',
            'normal_tool_call_text',
            'mismatch_sampling_type',
        ]
        if col in df.columns
    ]
    out = df[['user_prompt', 'tool_call_text', 'label'] + metadata_cols].copy() #원본 데이터프레임에서 필요한 칼럼만 뽑아서 out이라는 새 데이터프레임 생성
    out['user_prompt'] = out['user_prompt'].fillna('').astype(str).map(normalize_text) #user_promt 칼럼 정리
    out['tool_call_text'] = out['tool_call_text'].fillna('').astype(str).map(normalize_text) #tool_call_text 칼럼 정리
    out['label'] = out['label'].astype(int) #label값을 정수형으로 변환

    model = get_embedding_model() #잌베딩 모델 불러오기
    prompt_frame, prompt_feature_cols = encode_unique_text_column(out['user_prompt'], 'prompt', model) #프롬프트 칼럼 임베딩
    tool_frame, tool_feature_cols = encode_unique_text_column(out['tool_call_text'], 'tool', model) #도구이름 칼럼 임베딩

    prompt_matrix = prompt_frame.to_numpy(dtype=np.float32) #프롬프트 임베딩 df를 NumPy배열로 바꿈
    tool_matrix = tool_frame.to_numpy(dtype=np.float32) #도구호출 임베딩도 NumPy배열로 바꿈
    cosine_similarity = np.sum(prompt_matrix * tool_matrix, axis=1).astype(np.float32) #코사인 유사도 계산
    delta_matrix = (prompt_matrix - tool_matrix).astype(np.float32) #논문에서 사용하는 핵심 특징 벡터

    delta_feature_cols = [f'delta_emb_{index:03d}' for index in range(delta_matrix.shape[1])] #각 차원에 대한 컬럼 이름 생성
    delta_frame = pd.DataFrame(delta_matrix, columns=delta_feature_cols, index=out.index, dtype=np.float32) #df로 변환

    out = pd.concat([out, delta_frame], axis=1) #기존 out df에 delta embedding을 옆으로 붙임
    out['prompt_tool_cosine_similarity'] = cosine_similarity #코사인 유사도 값 추가
    out[delta_feature_cols] = out[delta_feature_cols].fillna(0) #결측값은 0으로 채움
    return out, delta_feature_cols



def group_train_valid_test_split( #source_prompt_hash 같은 그룹 기준을 유지하면서 trian/valid/test를 나누는 함수. 데이터 누수 방지용.
    df,
    group_col=SPLIT_GROUP_COL,
    label_col='label',
    train_ratio=TRAIN_RATIO,
    valid_ratio=VALID_RATIO,
    test_ratio=TEST_RATIO,
    random_seed=RANDOM_SEED,
    max_attempts=30,
):
    if not np.isclose(train_ratio + valid_ratio + test_ratio, 1.0): #train, valid, test 비율의 합이 1.0인지 확인
        raise ValueError('train_ratio + valid_ratio + test_ratio must sum to 1.0') #에러 처리

    groups = df[group_col].astype(str) #group split에 사용할 그룹칼럼을 문자열로 변환
    inner_valid_ratio = valid_ratio / (train_ratio + valid_ratio) #먼저 전체 데이터에서 test를 떼어낸 뒤, 남은 train_valid 데이터 안에서 valid를 얼마나 떼어낼지 계산

    for offset in range(max_attempts): #최대 시도 제한
        split_seed = random_seed + offset #매 시도마다 random_seed를 조금씩 바꿈
        outer_splitter = GroupShuffleSplit(n_splits=1, test_size=test_ratio, random_state=split_seed) #trian,valid와 test로 나눔
        train_valid_idx, test_idx = next(outer_splitter.split(df, y=df[label_col], groups=groups)) #인덱스 생성

        train_valid_df = df.iloc[train_valid_idx].reset_index(drop=True) #train, valid로 사용할 데이터를 생성
        test_df = df.iloc[test_idx].reset_index(drop=True) #test 데이터 생성

        inner_groups = train_valid_df[group_col].astype(str) #trian과 valid를 나눌때 사용할 그룹 정보
        inner_splitter = GroupShuffleSplit(n_splits=1, test_size=inner_valid_ratio, random_state=split_seed) #trian과 valid로 나눔. test_size는 validation 비율.
        train_idx, valid_idx = next( #train과 valid로 나눌 인덱스 생성
            inner_splitter.split(train_valid_df, y=train_valid_df[label_col], groups=inner_groups)
        )

        train_df = train_valid_df.iloc[train_idx].reset_index(drop=True) #최종 trian 데이터 셋
        valid_df = train_valid_df.iloc[valid_idx].reset_index(drop=True) #최종 validation 데이터 셋

        if all(split_df[label_col].nunique() == 2 for split_df in [train_df, valid_df, test_df]): #각각에 label이 2종류 모두 있는지 확인
            return train_df, valid_df, test_df, split_seed #조건 만족하면 df와 seed 반환

    raise ValueError('Could not create a split where every partition contains both labels.')


def print_split_summary(split_name, df, label_col='label'): #데이터 분할 결과를 요약해서 출력
    counts = df[label_col].value_counts().sort_index().to_dict()
    print(f'{split_name:<5} rows={len(df):>8} | label counts={counts}')


def evaluate_thresholds(y_true, y_scores, thresholds): #여러 threshold 후보를 평가해서 각 지표를 계산하는 함수
    y_true = np.asarray(y_true, dtype=int) #실제 label을 NumPy 배열로 변환하고 정수형으로 맞춤
    rows = [] #threshold별 평가 결과 저장할 리스트
    for threshold in thresholds:
        y_pred = (y_scores >= threshold).astype(int) # 모델 점수가 threshold이사잉면 1로 예측
        rows.append(
            {
                'threshold': float(threshold),
                'precision': float(precision_score(y_true, y_pred, zero_division=0)),
                'recall': float(recall_score(y_true, y_pred, zero_division=0)),
                'f1': float(f1_score(y_true, y_pred, zero_division=0)),
                'accuracy': float(accuracy_score(y_true, y_pred)),
            }
        )

    metrics_df = pd.DataFrame(rows) #결과 데이터프레임 생성
    best_row = metrics_df.sort_values( #가장 좋은 threshold 선택
        by=['f1', 'precision', 'recall', 'accuracy', 'threshold'],
        ascending=[False, False, False, False, True],
    ).iloc[0] #가장 좋은 threshold 결과 가져오기
    return metrics_df, best_row #모든 threshold별 성능표, 가장 좋은 threshold


def build_continuous_threshold_grid(scores, num=201): #모델이 출력한 score 값들을 기준으로 threshold 후보를 만드는 함수. 후보 201개 생성
    scores = np.asarray(scores, dtype=float) #scores를 NumPy 배열로 변환한고 float 타입으로 맞춤
    low = float(np.min(scores)) #가장 작은 값. 후보의 시작점
    high = float(np.max(scores)) #가장 큰 값. 후보의 끝점
    if np.isclose(low, high): #만약 최소값과최대값이 거의 같다면 범위를 여러 개로 나눌 필요가 없음.
        return np.array([low], dtype=float)
    return np.linspace(low, high, num=num) # 균등한 간격으로 201개 후보 생성


def plot_threshold_curve(metrics_df, best_threshold, output_path, title): #threshold별 지표 변화를 그래프로 저장
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(metrics_df['threshold'], metrics_df['f1'], label='F1')
    ax.plot(metrics_df['threshold'], metrics_df['precision'], label='Precision', alpha=0.8)
    ax.plot(metrics_df['threshold'], metrics_df['recall'], label='Recall', alpha=0.8)
    ax.axvline(best_threshold, color='red', linestyle='--', label=f'Best threshold={best_threshold:.4f}')
    ax.set_title(title)
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def plot_confusion_matrix_figure(cm, output_path, title): #confusion matrix를 이미지로 저장
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    image = ax.imshow(cm, cmap='Blues')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    labels = ['Pred 0', 'Pred 1']
    ax.set_xticks([0, 1], labels=labels)
    ax.set_yticks([0, 1], labels=['True 0', 'True 1'])
    ax.set_title(title)

    for row_idx in range(cm.shape[0]):
        for col_idx in range(cm.shape[1]):
            ax.text(col_idx, row_idx, int(cm[row_idx, col_idx]), ha='center', va='center', color='black')

    ax.set_xlabel('Prediction')
    ax.set_ylabel('Ground Truth')
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def plot_score_histogram(score_df, output_path, title): #모델 score분포를 label별로 히스토그램으로 저장
    fig, ax = plt.subplots(figsize=(8, 4))
    negatives = score_df.loc[score_df['label'] == 0, 'score']
    positives = score_df.loc[score_df['label'] == 1, 'score']
    ax.hist(negatives, bins=30, alpha=0.7, label='Label 0')
    ax.hist(positives, bins=30, alpha=0.7, label='Label 1')
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def make_lightgbm_classifiers(scale_pos_weight): #LighGBM 분류 모델 생성
    models = []
    for device_type in LIGHTGBM_DEVICE_CANDIDATES:
        params = {
            'objective': 'binary', #이진 분류 문제로 설정
            'n_estimators': 1000, #트리를 최대 1000개까지 사용
            'learning_rate': 0.05, #학습률
            'num_leaves': 31, #트리 하나가 가질 수 있는 leaf node의 수
            'subsample': 0.8, #과적합을 줄이기 위해 전체 데이터중 80%만 사용
            'colsample_bytree': 0.8, #feature중 80%만 사용
            'random_state': RANDOM_SEED, #실험 재현성을 위해 랜덤성 고정
            'scale_pos_weight': scale_pos_weight, #불균형 데이터에서 label1 가중치를 주기 위한 값
            'device_type': device_type,
        }
        if device_type == 'gpu':
            params.setdefault('max_bin', 255) 
        models.append((LGBMClassifier(**params), device_type))
    return models


def make_xgboost_classifier(scale_pos_weight): #XGBoost 분류 모델 생성
    models = []
    for device_name in XGBOOST_DEVICE_CANDIDATES:
        model = XGBClassifier(
            objective='binary:logistic', #이진 분류, label 1일 가능성을 score로 출력
            eval_metric='logloss', #평가 지표로 logloss를 사용. 예측 확률이 정답과 얼마나 어긋나는지를 보는 지표
            n_estimators=500, #트리를 최대 500개 사용
            learning_rate=0.05, #학습률 0.05로 설정
            max_depth=6, #과적합 방지를 위해 최대 깊이를 6으로 제한
            subsample=0.8, #데이터의 80%만 샘플링. 과적합 방지.
            colsample_bytree=0.8, #feature의 80%만 사용
            min_child_weight=1, #leaf node를 만들기 위한 최소 가중치 
            reg_lambda=1.0, #L2 정규화 강도. 과적합 방지.
            random_state=RANDOM_SEED, #실험 재현성을 위해 랜덤성 고정
            scale_pos_weight=scale_pos_weight, #불균형 데이터에서 label1 가중치를 주기 위한 값
            tree_method='hist', #히스토그램 기반 트리 학습. 대용량 데이터에서 빠르게 학습하기 위한 방식.
            device=device_name,
            n_jobs=-1, #가능한 CPU 코어를 최대한 사용
        )
        models.append((model, device_name))
    return models


def train_supervised_model(model_name, train_df, valid_df, feature_cols): #지도학습 모델을 학습시키는 함수
    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_df['label'].to_numpy(dtype=int)
    X_valid = valid_df[feature_cols].to_numpy(dtype=np.float32)
    y_valid = valid_df['label'].to_numpy(dtype=int)

    positive_count = int(np.sum(y_train == 1)) #비정상 샘플 개수
    negative_count = int(np.sum(y_train == 0)) #정상 샘플 개수
    scale_pos_weight = negative_count / max(positive_count, 1) #불균형 데이터셋에서 비정상 클래스가 무시되지 않도록 보정

    if model_name == 'lightgbm': #LightGBM 학습
        last_error = None
        for model, device_type in make_lightgbm_classifiers(scale_pos_weight=scale_pos_weight): #
            try: #에러 처리
                fit_start = time.perf_counter() #학습 시작시간 기록
                model.fit( #학습
                    X_train,
                    y_train,
                    eval_set=[(X_valid, y_valid)],
                    eval_metric='binary_logloss',
                    callbacks=[
                        early_stopping(stopping_rounds=50, verbose=True), #50반복 동안 좋아지지 않으면 학습을 조기 종료
                        log_evaluation(period=50),
                    ],
                )
                fit_seconds = time.perf_counter() - fit_start #학습 소요시간 계산
                metadata = {'model_device_type': device_type, 'fit_seconds': float(fit_seconds)}
                return model, metadata
            except Exception as exc:
                last_error = exc
                print(f'LightGBM fit failed on device={device_type}: {exc}')
        raise RuntimeError(f'LightGBM fit failed on every device candidate. Last error: {last_error}')

    if model_name == 'xgboost': #XGBoost 학습
        last_error = None
        for model, device_type in make_xgboost_classifier(scale_pos_weight=scale_pos_weight):
            try:
                fit_start = time.perf_counter() #학습 시작시간 기록
                model.fit( #학습
                    X_train,
                    y_train,
                    eval_set=[(X_valid, y_valid)],
                    verbose=False,
                )
                fit_seconds = time.perf_counter() - fit_start
                metadata = {'model_device_type': device_type, 'fit_seconds': float(fit_seconds)}
                return model, metadata
            except Exception as exc:
                last_error = exc
                print(f'XGBoost fit failed on device={device_type}: {exc}')
        raise RuntimeError(f'XGBoost fit failed on every device candidate. Last error: {last_error}')

    if model_name == 'random_forest': #Random Forest 학습
        model = RandomForestClassifier(
            n_estimators=400, #결정트리 400개 사용
            class_weight='balanced_subsample', #불균형 데이터 해소
            random_state=RANDOM_SEED, #재현성을 위해 랜덤시드 고정
            n_jobs=-1,
        )
        fit_start = time.perf_counter()
        model.fit(X_train, y_train) #학습
        fit_seconds = time.perf_counter() - fit_start #학습 소요시간 계산
        return model, {'model_device_type': 'cpu', 'fit_seconds': float(fit_seconds)}

    raise ValueError(f'Unsupported supervised model: {model_name}')


def build_split_scored_frame(train_df, valid_df, test_df, train_scores, valid_scores, test_scores): #tran, valid, test df와 각각의 예측 점수를 받아서 하나의 score df로 합치는 함수
    train_part = train_df[['user_prompt', 'tool_call_text', 'label']].copy()
    train_part['split'] = 'train'
    train_part['score'] = train_scores

    valid_part = valid_df[['user_prompt', 'tool_call_text', 'label']].copy()
    valid_part['split'] = 'valid'
    valid_part['score'] = valid_scores

    test_part = test_df[['user_prompt', 'tool_call_text', 'label']].copy()
    test_part['split'] = 'test'
    test_part['score'] = test_scores

    return pd.concat([train_part, valid_part, test_part], ignore_index=True)


def run_supervised_experiment(model_name, train_df, valid_df, test_df, feature_cols, run_root): #전체 실험 실행 함수
    model_dir = run_root / model_name #모델별 결과 저장 폴더 경로 생성
    model_dir.mkdir(parents=True, exist_ok=True) #폴더 생성

    model, model_metadata = train_supervised_model(model_name, train_df, valid_df, feature_cols) #모델 학습

    #train,valid, test를 X,y로 분리
    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_df['label'].to_numpy(dtype=int)
    X_valid = valid_df[feature_cols].to_numpy(dtype=np.float32)
    y_valid = valid_df['label'].to_numpy(dtype=int)
    X_test = test_df[feature_cols].to_numpy(dtype=np.float32)
    y_test = test_df['label'].to_numpy(dtype=int)

    #label 1일 확률 계산
    train_scores = model.predict_proba(X_train)[:, 1].astype(float)
    valid_scores = model.predict_proba(X_valid)[:, 1].astype(float)
    test_scores = model.predict_proba(X_test)[:, 1].astype(float)

    #valid 에서 최적 threshold 선택
    threshold_metrics_df, best_row = evaluate_thresholds(y_valid, valid_scores, DEFAULT_THRESHOLDS)
    best_threshold = float(best_row['threshold'])
    test_pred = (test_scores >= best_threshold).astype(int) #test에 최적 값 적용

    #성능 계산
    precision = float(precision_score(y_test, test_pred, zero_division=0))
    recall = float(recall_score(y_test, test_pred, zero_division=0))
    f1 = float(f1_score(y_test, test_pred, zero_division=0))
    accuracy = float(accuracy_score(y_test, test_pred))
    auroc = float(roc_auc_score(y_test, test_scores))
    auprc = float(average_precision_score(y_test, test_scores))
    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

    #score포함 데이터프레임 생성
    split_scored_df = build_split_scored_frame(train_df, valid_df, test_df, train_scores, valid_scores, test_scores)
    test_predictions_df = test_df[['user_prompt', 'tool_call_text', 'label']].copy()
    test_predictions_df['score'] = test_scores
    test_predictions_df['pred_label'] = test_pred

    #저장 경로 설정
    threshold_metrics_path = model_dir / 'valid_threshold_metrics.csv'
    split_scored_path = model_dir / 'split_scored.csv'
    test_predictions_path = model_dir / 'test_predictions.csv'
    confusion_path = model_dir / 'test_confusion_matrix.png'
    threshold_curve_path = model_dir / 'valid_f1_curve.png'
    histogram_path = model_dir / 'test_score_histogram.png'
    summary_path = model_dir / 'summary.json'

    #CSV 저장
    threshold_metrics_df.to_csv(threshold_metrics_path, index=False, encoding='utf-8-sig')
    split_scored_df.to_csv(split_scored_path, index=False, encoding='utf-8-sig')
    test_predictions_df.to_csv(test_predictions_path, index=False, encoding='utf-8-sig')

    #그림 저장
    plot_confusion_matrix_figure(cm, confusion_path, f'Test Confusion Matrix - {model_name}')
    plot_threshold_curve(threshold_metrics_df, best_threshold, threshold_curve_path, f'Validation Threshold Sweep - {model_name}')
    plot_score_histogram(test_predictions_df[['label', 'score']], histogram_path, f'Test Score Histogram - {model_name}')

    #최종 실험 결과 딕셔너리로 정리
    summary = {
        'model_name': model_name,
        'score_kind': 'probability',
        'best_validation_threshold': best_threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'auroc': auroc,
        'auprc': auprc,
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'model_device_type': model_metadata['model_device_type'],
        'embedding_device': EMBEDDING_DEVICE,
        'fit_seconds': model_metadata['fit_seconds'],
        'output_dir': str(model_dir),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8') #json으로 저장

    return {
        'model_name': model_name,
        'model': model,
        'feature_cols': feature_cols,
        'score_kind': 'probability',
        'summary': summary,
        'test_predictions': test_predictions_df,
        'split_scored': split_scored_df,
        'output_dir': model_dir,
        'summary_path': summary_path,
    }

#Isolation Forest를 정상데이터만으로 학습시키는 함수
def train_isolation_forest(train_df, feature_cols):
    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_df['label'].to_numpy(dtype=int)
    X_train_normal = X_train[y_train == 0] #정상 샘플만 선택

    model = IsolationForest(
        n_estimators=400, #트리 400개
        contamination='auto',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    fit_start = time.perf_counter()
    model.fit(X_train_normal) #정상 데이터만으로 학습
    fit_seconds = time.perf_counter() - fit_start
    return model, {'model_device_type': 'cpu', 'fit_seconds': float(fit_seconds)}

#Isolation Forest 모델 실험 실행
def run_isolation_forest_experiment(train_df, valid_df, test_df, feature_cols, run_root):
    model_name = 'isolation_forest'
    model_dir = run_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True) #결과 저장 폴더 생성

    model, model_metadata = train_isolation_forest(train_df, feature_cols) #학습

    X_train = train_df[feature_cols].to_numpy(dtype=np.float32) #train 데이터의 feature
    y_train = train_df['label'].to_numpy(dtype=int)
    X_valid = valid_df[feature_cols].to_numpy(dtype=np.float32) #valid의 feature
    y_valid = valid_df['label'].to_numpy(dtype=int) #valid의 라벨
    X_test = test_df[feature_cols].to_numpy(dtype=np.float32) #test의 feature
    y_test = test_df['label'].to_numpy(dtype=int) #test의 라벨

    #원래는 정상일 수록 점수가 높지만, 점수가 높을 수록 비정상일 가능성이 높은게 목적이기 때문에 -를 붙여 방향을 뒤집음
    train_scores = (-model.score_samples(X_train)).astype(float)
    valid_scores = (-model.score_samples(X_valid)).astype(float)
    test_scores = (-model.score_samples(X_test)).astype(float)

    threshold_grid = build_continuous_threshold_grid(valid_scores, num=201) #valid score 범위에 맞춰 threshold 후보 생성
    threshold_metrics_df, best_row = evaluate_thresholds(y_valid, valid_scores, threshold_grid) # 최적의 threshold 선택
    best_threshold = float(best_row['threshold'])
    test_pred = (test_scores >= best_threshold).astype(int) #test 데이터에 적용

    #성능 계산
    precision = float(precision_score(y_test, test_pred, zero_division=0))
    recall = float(recall_score(y_test, test_pred, zero_division=0))
    f1 = float(f1_score(y_test, test_pred, zero_division=0))
    accuracy = float(accuracy_score(y_test, test_pred))
    auroc = float(roc_auc_score(y_test, test_scores)) #score 자체가 label0과 label1을 얼마나 잘 구분하는지 보는 지표
    auprc = float(average_precision_score(y_test, test_scores))
    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

    #score 데이터프레임 생성
    split_scored_df = build_split_scored_frame(train_df, valid_df, test_df, train_scores, valid_scores, test_scores)
    test_predictions_df = test_df[['user_prompt', 'tool_call_text', 'label']].copy()
    test_predictions_df['score'] = test_scores
    test_predictions_df['pred_label'] = test_pred

    #저장경로 설정
    threshold_metrics_path = model_dir / 'valid_threshold_metrics.csv'
    split_scored_path = model_dir / 'split_scored.csv'
    test_predictions_path = model_dir / 'test_predictions.csv'
    confusion_path = model_dir / 'test_confusion_matrix.png'
    threshold_curve_path = model_dir / 'valid_f1_curve.png'
    histogram_path = model_dir / 'test_score_histogram.png'
    summary_path = model_dir / 'summary.json'

    #CSV 저장
    threshold_metrics_df.to_csv(threshold_metrics_path, index=False, encoding='utf-8-sig')
    split_scored_df.to_csv(split_scored_path, index=False, encoding='utf-8-sig')
    test_predictions_df.to_csv(test_predictions_path, index=False, encoding='utf-8-sig')

    #그림 저장
    plot_confusion_matrix_figure(cm, confusion_path, 'Test Confusion Matrix - isolation_forest')
    plot_threshold_curve(threshold_metrics_df, best_threshold, threshold_curve_path, 'Validation Threshold Sweep - isolation_forest')
    plot_score_histogram(test_predictions_df[['label', 'score']], histogram_path, 'Test Score Histogram - isolation_forest')

    #summary 생성
    summary = {
        'model_name': model_name,
        'score_kind': 'anomaly_score',
        'best_validation_threshold': best_threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'auroc': auroc,
        'auprc': auprc,
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'model_device_type': model_metadata['model_device_type'],
        'embedding_device': EMBEDDING_DEVICE,
        'fit_seconds': model_metadata['fit_seconds'],
        'output_dir': str(model_dir),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    return {
        'model_name': model_name,
        'model': model,
        'feature_cols': feature_cols,
        'score_kind': 'anomaly_score',
        'summary': summary,
        'test_predictions': test_predictions_df,
        'split_scored': split_scored_df,
        'output_dir': model_dir,
        'summary_path': summary_path,
    }

#코사인 유사도 baseline 실험
def run_cosine_similarity_experiment(train_df, valid_df, test_df, run_root):
    model_name = 'cosine_similarity'
    model_dir = run_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    #정답 라벨 가져오기
    y_valid = valid_df['label'].to_numpy(dtype=int)
    y_test = test_df['label'].to_numpy(dtype=int)

    #score 계산
    #원래는 유사도가 높으면 정상일 확률이 높은데 원하는 방향은 score가 높을때 mismatch의 확률이 높은 것이니 1-유사도를 계산.
    train_scores = (1.0 - train_df['prompt_tool_cosine_similarity'].to_numpy(dtype=float)).astype(float)
    valid_scores = (1.0 - valid_df['prompt_tool_cosine_similarity'].to_numpy(dtype=float)).astype(float)
    test_scores = (1.0 - test_df['prompt_tool_cosine_similarity'].to_numpy(dtype=float)).astype(float)

    #threshold 후보생성하고 결정
    threshold_grid = build_continuous_threshold_grid(valid_scores, num=201)
    threshold_metrics_df, best_row = evaluate_thresholds(y_valid, valid_scores, threshold_grid)
    best_threshold = float(best_row['threshold'])
    test_pred = (test_scores >= best_threshold).astype(int)

    #성능 계산
    precision = float(precision_score(y_test, test_pred, zero_division=0))
    recall = float(recall_score(y_test, test_pred, zero_division=0))
    f1 = float(f1_score(y_test, test_pred, zero_division=0))
    accuracy = float(accuracy_score(y_test, test_pred))
    auroc = float(roc_auc_score(y_test, test_scores))
    auprc = float(average_precision_score(y_test, test_scores))
    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

    #결과 데이터프레임 생성
    split_scored_df = build_split_scored_frame(train_df, valid_df, test_df, train_scores, valid_scores, test_scores)
    test_predictions_df = test_df[['user_prompt', 'tool_call_text', 'label']].copy()
    test_predictions_df['cosine_similarity'] = test_df['prompt_tool_cosine_similarity'].to_numpy(dtype=float)
    test_predictions_df['score'] = test_scores
    test_predictions_df['pred_label'] = test_pred

    #저장 경로 설정
    threshold_metrics_path = model_dir / 'valid_threshold_metrics.csv'
    split_scored_path = model_dir / 'split_scored.csv'
    test_predictions_path = model_dir / 'test_predictions.csv'
    confusion_path = model_dir / 'test_confusion_matrix.png'
    threshold_curve_path = model_dir / 'valid_f1_curve.png'
    histogram_path = model_dir / 'test_score_histogram.png'
    summary_path = model_dir / 'summary.json'

    #CSV 저장
    threshold_metrics_df.to_csv(threshold_metrics_path, index=False, encoding='utf-8-sig')
    split_scored_df.to_csv(split_scored_path, index=False, encoding='utf-8-sig')
    test_predictions_df.to_csv(test_predictions_path, index=False, encoding='utf-8-sig')

    #그림 저장
    plot_confusion_matrix_figure(cm, confusion_path, 'Test Confusion Matrix - cosine_similarity')
    plot_threshold_curve(threshold_metrics_df, best_threshold, threshold_curve_path, 'Validation Threshold Sweep - cosine_similarity')
    plot_score_histogram(test_predictions_df[['label', 'score']], histogram_path, 'Test Score Histogram - cosine_similarity')

    #summary 생성
    summary = {
        'model_name': model_name,
        'score_kind': 'cosine_mismatch_score',
        'score_definition': 'score = 1 - prompt_tool_cosine_similarity; higher means more mismatch-like',
        'best_validation_threshold': best_threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'auroc': auroc,
        'auprc': auprc,
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'model_device_type': 'none',
        'embedding_device': EMBEDDING_DEVICE,
        'fit_seconds': 0.0,
        'output_dir': str(model_dir),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    return {
        'model_name': model_name,
        'model': None,
        'feature_cols': ['prompt_tool_cosine_similarity'],
        'score_kind': 'cosine_mismatch_score',
        'summary': summary,
        'test_predictions': test_predictions_df,
        'split_scored': split_scored_df,
        'output_dir': model_dir,
        'summary_path': summary_path,
    }

#단일 요청에 대한 실시간 추론시나리오 구현
#latency 계산용
def build_single_row_feature_frame(user_prompt, tool_call_text, feature_cols):
    model = get_embedding_model()
    prompt_text = normalize_text(user_prompt)
    tool_text = normalize_text(tool_call_text)

    synchronize_if_needed()
    embedding_start = time.perf_counter()
    prompt_emb = model.encode(
        [prompt_text],
        batch_size=1,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    tool_emb = model.encode(
        [tool_text],
        batch_size=1,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    synchronize_if_needed()
    embedding_ms = (time.perf_counter() - embedding_start) * 1000.0

    cosine_similarity = float(np.dot(prompt_emb[0], tool_emb[0]))
    delta_vec = (prompt_emb[0] - tool_emb[0]).astype(np.float32)
    row_values = {f'delta_emb_{index:03d}': float(value) for index, value in enumerate(delta_vec)}
    row_values['prompt_tool_cosine_similarity'] = cosine_similarity

    feature_frame = pd.DataFrame([{column: row_values[column] for column in feature_cols}], columns=feature_cols)
    feature_frame = feature_frame.astype(np.float32)
    return feature_frame, embedding_ms


#샘플 1개에 대해 score를 계산하는 함수
#latency 계산용
def score_single_row(model, score_kind, feature_frame):
    feature_array = feature_frame.to_numpy(dtype=np.float32)
    if score_kind == 'probability':
        return float(model.predict_proba(feature_array)[0, 1])
    if score_kind == 'anomaly_score':
        return float((-model.score_samples(feature_array))[0])
    if score_kind == 'cosine_mismatch_score':
        return float(1.0 - feature_frame['prompt_tool_cosine_similarity'].iloc[0])
    raise ValueError(f'Unsupported score_kind: {score_kind}')

#새로운 샘플 1개를 넣었을 때 예측 결과와 지연시간을 함께 측정하는 함수
def predict_single_row_with_latency(user_prompt, tool_call_text, result):
    total_start = time.perf_counter()
    feature_frame, embedding_ms = build_single_row_feature_frame(
        user_prompt=user_prompt,
        tool_call_text=tool_call_text,
        feature_cols=result['feature_cols'],
    )

    model_start = time.perf_counter()
    score = score_single_row(
        model=result['model'],
        score_kind=result['score_kind'],
        feature_frame=feature_frame,
    )
    model_inference_ms = (time.perf_counter() - model_start) * 1000.0
    end_to_end_ms = (time.perf_counter() - total_start) * 1000.0

    pred_label = int(score >= result['summary']['best_validation_threshold'])
    return {
        'pred_score': float(score),
        'pred_label': pred_label,
        'embedding_ms': float(embedding_ms),
        'model_inference_ms': float(model_inference_ms),
        'end_to_end_ms': float(end_to_end_ms),
    }

#테스트 샘플 중 일부를 뽑아서 단일 요청 기준 지연시간 측정하는 함수
def benchmark_latency(result, sample_size=LATENCY_SAMPLE_SIZE):
    latency_input_df = ( #latency 측정에 사용할 test 샘플 뽑기
        result['test_predictions'][['user_prompt', 'tool_call_text', 'label']]
        .sample(n=min(sample_size, len(result['test_predictions'])), random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    latency_rows = []
    for row in latency_input_df.itertuples(index=False):
        latency_result = predict_single_row_with_latency( #샘플 하나에 대해 실제 예측과 지연시간 측정
            user_prompt=row.user_prompt,
            tool_call_text=row.tool_call_text,
            result=result,
        )
        latency_rows.append( # 샘플 하나의 결과를 추가
            {
                'user_prompt': row.user_prompt,
                'tool_call_text': row.tool_call_text,
                'true_label': int(row.label),
                **latency_result,
            }
        )

    latency_samples_df = pd.DataFrame(latency_rows) #df생성
    latency_samples_path = result['output_dir'] / 'latency_samples.csv' #저장 경로 설정
    latency_summary_path = result['output_dir'] / 'latency_summary.json'

    latency_samples_df.to_csv(latency_samples_path, index=False, encoding='utf-8-sig') #csv로 저장

    #latency 요약 계산
    latency_summary = {
        'sample_size': int(len(latency_samples_df)),
        'embedding_device': EMBEDDING_DEVICE,
        'model_device_type': result['summary']['model_device_type'],
        'embedding_ms_mean': float(latency_samples_df['embedding_ms'].mean()),
        'embedding_ms_median': float(latency_samples_df['embedding_ms'].median()),
        'embedding_ms_p95': float(np.percentile(latency_samples_df['embedding_ms'], 95)),
        'model_inference_ms_mean': float(latency_samples_df['model_inference_ms'].mean()),
        'model_inference_ms_median': float(latency_samples_df['model_inference_ms'].median()),
        'model_inference_ms_p95': float(np.percentile(latency_samples_df['model_inference_ms'], 95)),
        'end_to_end_ms_mean': float(latency_samples_df['end_to_end_ms'].mean()),
        'end_to_end_ms_median': float(latency_samples_df['end_to_end_ms'].median()),
        'end_to_end_ms_p95': float(np.percentile(latency_samples_df['end_to_end_ms'], 95)),
        'latency_samples_path': str(latency_samples_path),
    }
    latency_summary_path.write_text(json.dumps(latency_summary, ensure_ascii=False, indent=2), encoding='utf-8')

    result['summary']['latency_summary_path'] = str(latency_summary_path)
    result['summary']['latency_summary'] = latency_summary

    summary_payload = json.loads(result['summary_path'].read_text(encoding='utf-8'))
    summary_payload['latency_summary_path'] = str(latency_summary_path)
    summary_payload['latency_summary'] = latency_summary
    result['summary_path'].write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

    latency_report_df = pd.DataFrame( #보고용 df 생성성
        [
            {
                'metric': 'embedding_ms',
                'mean': latency_summary['embedding_ms_mean'],
                'median': latency_summary['embedding_ms_median'],
                'p95': latency_summary['embedding_ms_p95'],
            },
            {
                'metric': 'model_inference_ms',
                'mean': latency_summary['model_inference_ms_mean'],
                'median': latency_summary['model_inference_ms_median'],
                'p95': latency_summary['model_inference_ms_p95'],
            },
            {
                'metric': 'end_to_end_ms',
                'mean': latency_summary['end_to_end_ms_mean'],
                'median': latency_summary['end_to_end_ms_median'],
                'p95': latency_summary['end_to_end_ms_p95'],
            },
        ]
    )
    return latency_samples_df, latency_summary, latency_report_df


## Cell 4. Load the full mismatch dataset


In [ ]:
raw_df = pd.read_csv(DATA_PATH) #데이터셋 불러오기

required_cols = ['user_prompt', 'tool_call_text', 'label', SPLIT_GROUP_COL] #필수 칼럼 확인
missing_cols = [col for col in required_cols if col not in raw_df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}') #빠진 칼럼 있으면 에러 발생

raw_df['label'] = raw_df['label'].astype(int) #label 칼럼을 정수형으로 변환
label_counts = raw_df['label'].value_counts().sort_index().to_dict() #개수 계산
if label_counts.get(0, 0) < NORMAL_SAMPLE_SIZE: #정상 샘플 수가 충분한지 확인
    raise ValueError(f'Not enough label 0 rows: {label_counts.get(0, 0)} < {NORMAL_SAMPLE_SIZE}')
if label_counts.get(1, 0) < MAX_NEGATIVE_SAMPLE_SIZE: #비정상 샘플 수가 충분한지 확인
    raise ValueError(f'Not enough label 1 rows: {label_counts.get(1, 0)} < {MAX_NEGATIVE_SAMPLE_SIZE}')

label0_groups = set(raw_df.loc[raw_df['label'] == 0, SPLIT_GROUP_COL].astype(str)) #label0에서 split_group_col을 값을 가져와서 set으로 만듦
label1_groups = set(raw_df.loc[raw_df['label'] == 1, SPLIT_GROUP_COL].astype(str)) #label1에서 split_group_col을 값을 가져와서 set으로 만듦

#정보 출력
print('Dataset shape          :', raw_df.shape)
print('Label counts           :', label_counts)
print('Unique split groups    :', raw_df[SPLIT_GROUP_COL].nunique())
print('Label 0 groups         :', len(label0_groups))
print('Label 1 groups         :', len(label1_groups))
print('Groups shared by labels:', len(label0_groups & label1_groups))
print('Columns                :', raw_df.columns.tolist())

#미리보기용 칼럼 선택
preview_cols = [
    col
    for col in [
        'user_prompt',
        'tool_call_text',
        'label',
        'prompt_source_file',
        'prompt_source_toolkit',
        'tool_source_toolkit',
        'source_prompt_hash',
        'normal_tool_call_text',
        'mismatch_sampling_type',
    ]
    if col in raw_df.columns
]
raw_df[preview_cols].head(5)


## Cell 5. Build delta embedding features and cosine baseline score


In [ ]:
feature_df, feature_cols = build_delta_feature_frame(raw_df) #원본 데이터셋을 임베딩 기반 feature 데이터셋으로 변환

#결과 확인
print('Total feature count  :', len(feature_cols))
print('Delta embedding dims :', len(feature_cols))
print('Cosine baseline col  :', 'prompt_tool_cosine_similarity')
print('Feature frame shape  :', feature_df[feature_cols].shape)
print('Feature frame labels :', feature_df['label'].value_counts().sort_index().to_dict())


## Cell 6. Feature list check


In [ ]:
feature_summary_df = pd.DataFrame( #feature_cols에 들어있는 feature 이름들을 df로 정리리
    {
        'feature_name': feature_cols,
        'feature_group': ['delta_embedding' for _ in feature_cols],
    }
)

print('Feature group counts:')
display(feature_summary_df['feature_group'].value_counts().rename_axis('feature_group').reset_index(name='count'))

print('First 20 feature names:')
display(feature_summary_df.head(20))

print('Last 20 feature names:')
display(feature_summary_df.tail(20))


## Cell 7. Sampled delta embedding distribution


In [ ]:
analysis_parts = []
for label_value in [0, 1]:
    subset = feature_df[feature_df['label'] == label_value]
    sample_size = min(ANALYSIS_SAMPLE_PER_LABEL, len(subset))
    analysis_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + label_value))

analysis_df = (
    pd.concat(analysis_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)
analysis_delta_matrix = analysis_df[feature_cols].to_numpy(dtype=np.float32)
analysis_labels = analysis_df['label'].to_numpy(dtype=int)
analysis_delta_norms = np.linalg.norm(analysis_delta_matrix, axis=1).astype(np.float32)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
delta_pca_2d = pca.fit_transform(analysis_delta_matrix) #차이벡터를 2차원 pca로 표현
plot_df = analysis_df[['label']].copy()
plot_df['delta_norm'] = analysis_delta_norms
plot_df['pca_x'] = delta_pca_2d[:, 0]
plot_df['pca_y'] = delta_pca_2d[:, 1]

pca_plot_path = RUN_ROOT / 'delta_embedding_pca_sampled.png'
norm_plot_path = RUN_ROOT / 'delta_embedding_norm_histogram_sampled.png'

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

#PCA 산점도 그리기
for label_value, color, label_name in [(0, '#1f77b4', 'Normal (0)'), (1, '#d62728', 'Mismatch (1)')]:
    subset = plot_df[plot_df['label'] == label_value]
    axes[0].scatter(
        subset['pca_x'],
        subset['pca_y'],
        s=12,
        alpha=0.55,
        c=color,
        label=label_name,
    )

axes[0].set_title('Delta embedding PCA (sampled)')
axes[0].set_xlabel('PCA 1')
axes[0].set_ylabel('PCA 2')
axes[0].legend()
axes[0].grid(alpha=0.2)

#L2 norm 히스토그램 그리기
for label_value, color, label_name in [(0, '#1f77b4', 'Normal (0)'), (1, '#d62728', 'Mismatch (1)')]:
    subset = plot_df[plot_df['label'] == label_value]
    axes[1].hist(
        subset['delta_norm'],
        bins=40,
        alpha=0.55,
        color=color,
        label=label_name,
        density=True,
    )

axes[1].set_title('Delta embedding L2 norm distribution (sampled)')
axes[1].set_xlabel('L2 norm of (prompt_emb - tool_emb)')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(pca_plot_path, dpi=200, bbox_inches='tight')
plt.savefig(norm_plot_path, dpi=200, bbox_inches='tight')
plt.show()

variance_ratio = pca.explained_variance_ratio_
print('Analysis sample shape:', analysis_df.shape)
print('PCA plot path        :', pca_plot_path)
print('Norm plot path       :', norm_plot_path)
print('PCA variance         :', [round(float(v), 4) for v in variance_ratio])
print('Label-wise delta norm summary:')
display(
    plot_df.groupby('label')['delta_norm']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .reset_index()
)


## Cell 8. Sampled centroid and projection analysis


In [ ]:
normal_matrix = analysis_delta_matrix[analysis_labels == 0]
negative_matrix = analysis_delta_matrix[analysis_labels == 1]

#라벨별 중심점 계산
normal_centroid = normal_matrix.mean(axis=0)
negative_centroid = negative_matrix.mean(axis=0)
centroid_delta = negative_centroid - normal_centroid #정상에서 비정상 쪽으로 향하는 방향
centroid_delta_norm = float(np.linalg.norm(centroid_delta)) # 두 중심점 사이의 거리

normal_centroid_norm = float(np.linalg.norm(normal_centroid))
negative_centroid_norm = float(np.linalg.norm(negative_centroid))
centroid_cosine = float(
    np.dot(normal_centroid, negative_centroid)
    / max(normal_centroid_norm * negative_centroid_norm, 1e-12) # 정상 중심과 비정상 중심 사이의 코사인 유사도를 계산
)

#class 내부 퍼짐 정도 계산
normal_spread = np.linalg.norm(normal_matrix - normal_centroid, axis=1)
negative_spread = np.linalg.norm(negative_matrix - negative_centroid, axis=1)

if centroid_delta_norm > 0:
    separation_axis = centroid_delta / centroid_delta_norm
else:
    separation_axis = np.zeros_like(centroid_delta)

#각 샘플을 분리 축에 투영
projection_scores = analysis_delta_matrix @ separation_axis
projection_df = pd.DataFrame(
    {
        'label': analysis_labels,
        'projection_score': projection_scores,
    }
)

#히스토그램 그리기
projection_plot_path = RUN_ROOT / 'delta_embedding_projection_histogram_sampled.png'
fig, ax = plt.subplots(figsize=(10, 5))
for label_value, color, label_name in [(0, '#1f77b4', 'Normal (0)'), (1, '#d62728', 'Mismatch (1)')]:
    subset = projection_df[projection_df['label'] == label_value]
    ax.hist(
        subset['projection_score'],
        bins=40,
        alpha=0.55,
        density=True,
        color=color,
        label=label_name,
    )
ax.set_title('Projection on centroid-difference axis (sampled)')
ax.set_xlabel('Projection score')
ax.set_ylabel('Density')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(projection_plot_path, dpi=200, bbox_inches='tight')
plt.show()

#실루엣 계산용 샘플링
silhouette_parts = []
for label_value in [0, 1]:
    subset = analysis_df[analysis_df['label'] == label_value]
    sample_size = min(SILHOUETTE_SAMPLE_PER_LABEL, len(subset))
    silhouette_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + 100 + label_value))
silhouette_df = pd.concat(silhouette_parts, ignore_index=True)
silhouette_matrix = silhouette_df[feature_cols].to_numpy(dtype=np.float32)
silhouette_labels = silhouette_df['label'].to_numpy(dtype=int)

#실루엣 스코어 계산
try:
    silhouette = float(silhouette_score(silhouette_matrix, silhouette_labels, metric='euclidean'))
except Exception as exc:
    silhouette = None
    print('Silhouette score could not be computed:', exc)

#분리도 요약표 만들기
separation_summary_df = pd.DataFrame(
    [
        {'metric': 'centroid_l2_distance', 'value': centroid_delta_norm},
        {'metric': 'centroid_cosine_similarity', 'value': centroid_cosine},
        {'metric': 'normal_within_class_mean_l2', 'value': float(normal_spread.mean())},
        {'metric': 'mismatch_within_class_mean_l2', 'value': float(negative_spread.mean())},
        {'metric': 'normal_projection_mean', 'value': float(projection_df.loc[projection_df['label'] == 0, 'projection_score'].mean())},
        {'metric': 'mismatch_projection_mean', 'value': float(projection_df.loc[projection_df['label'] == 1, 'projection_score'].mean())},
        {'metric': 'silhouette_score_sampled', 'value': silhouette},
    ]
)

#결과 출력
print('Projection plot path :', projection_plot_path)
print('Silhouette sample    :', silhouette_df.shape)
print('Separation summary:')
display(separation_summary_df)

print('Projection quantiles by label:')
display(
    projection_df.groupby('label')['projection_score']
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    .unstack()
    .reset_index()
)


## Cell 9. Build fixed-normal ratio samples


In [ ]:
def required_negative_count(config, normal_count=NORMAL_SAMPLE_SIZE): #정상 샘플 수를 기준으로 필요한 비정상 샘플 수를 계산
    return int(round(normal_count * config['negative_parts'] / config['normal_parts']))


def make_pair_key(df): #promt-tool 쌍을 식별하는  key 생성
    return df['user_prompt'].astype(str) + '/x1f' + df['tool_call_text'].astype(str)


def build_ratio_sample_frames(df): #비율별 샘플 데이터셋 생성 
    normal_all = df[df['label'] == 0].copy()
    negative_all = df[df['label'] == 1].copy()

    base_normal_df = normal_all.sample(n=NORMAL_SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True) #normal sample size만큼 뽑음. 논문에선 10만개.
    selected_groups = set(base_normal_df[SPLIT_GROUP_COL].astype(str)) #정상 샘플들이 어떤 source_promt_hash 그룹에 속하는지 모음

    negative_candidates = negative_all[  #비정상 샘플 중에서 선택된 정상 샘플과 같은 그룹에 속하는 것만 후보로 둠
        negative_all[SPLIT_GROUP_COL].astype(str).isin(selected_groups)
    ].copy()
    if len(negative_candidates) < MAX_NEGATIVE_SAMPLE_SIZE: #비정상 후보 수가 충분한지 확인
        raise ValueError(
            f'Not enough negative candidates from selected normal groups: '
            f'{len(negative_candidates)} < {MAX_NEGATIVE_SAMPLE_SIZE}'
        )

    negative_pool_df = ( #비정상 pool 10만개 생성
        negative_candidates
        .sample(n=MAX_NEGATIVE_SAMPLE_SIZE, random_state=RANDOM_SEED + 1000)
        .reset_index(drop=True)
    )

    raw_output_cols = [ #저장할 원 본 칼럼 목록 선택
        col
        for col in [
            'user_prompt',
            'tool_call_text',
            'label',
            'prompt_source_file',
            'prompt_source_toolkit',
            'tool_source_file',
            'tool_source_toolkit',
            'source_prompt_hash',
            'normal_tool_call_text',
            'mismatch_sampling_type',
        ]
        if col in df.columns
    ]

    sample_frames = {}
    plan_rows = []
    normal_pair_keys = set(make_pair_key(base_normal_df))

    for idx, config in enumerate(RATIO_CONFIGS): #비율별 데이터셋 생성
        negative_count = required_negative_count(config) #현재 비율에서 필요한 비정상 샘플 개수를 계산
        ratio_negative_df = negative_pool_df.head(negative_count).copy() #비정상 샘플 선택
        ratio_df = pd.concat([base_normal_df.copy(), ratio_negative_df], ignore_index=True) #정상과 해당 비율의 비정상 샘플을 합침
        ratio_df = ratio_df.sample(frac=1, random_state=RANDOM_SEED + idx).reset_index(drop=True) #합쳐진 데이터를 랜덤하게 섞음

        negative_pair_keys = set(make_pair_key(ratio_negative_df)) #정상 샘플과 비정상 샘플 사이에 동일한 쌍이 있는지 확인
        pair_overlap_count = len(normal_pair_keys & negative_pair_keys)
        sample_path = RUN_ROOT / f"sampled_dataset_ratio_{config['ratio_name']}.csv" #비율별 샘플 csv 저장
        ratio_df[raw_output_cols].to_csv(sample_path, index=False, encoding='utf-8-sig') 

        #샘플링 요약정보 저장
        label_counts = ratio_df['label'].value_counts().sort_index().to_dict() 
        plan_rows.append(
            {
                'ratio_name': config['ratio_name'],
                'ratio_label': config['ratio_label'],
                'normal_count': int(label_counts.get(0, 0)),
                'negative_count': int(label_counts.get(1, 0)),
                'total_count': int(len(ratio_df)),
                'positive_rate': float(label_counts.get(1, 0) / len(ratio_df)),
                'actual_normal_to_negative': float(label_counts.get(0, 0) / max(label_counts.get(1, 0), 1)),
                'unique_source_prompt_hash': int(ratio_df[SPLIT_GROUP_COL].nunique()),
                'normal_negative_pair_overlap': int(pair_overlap_count),
                'sample_path': str(sample_path),
            }
        )
        sample_frames[config['ratio_name']] = ratio_df

    return sample_frames, pd.DataFrame(plan_rows)


ratio_sample_frames, ratio_sampling_plan_df = build_ratio_sample_frames(feature_df) #feature_df를 입력으로 넣어 비율별 샘플 데이터셋을 생성
ratio_sampling_plan_path = RUN_ROOT / 'ratio_sampling_plan.csv' #샘플링 요약표 저장경로 생성
ratio_sampling_plan_df.to_csv(ratio_sampling_plan_path, index=False, encoding='utf-8-sig') # 샘플링 계획표 CSV로 저장

print('Ratio sampling plan path:', ratio_sampling_plan_path) #출력
display(ratio_sampling_plan_df)


## Cell 10. Train and evaluate models for each ratio


In [ ]:
ratio_experiment_results = {} #비율별 모델별 실험 결과를 저장할 딕셔너리
ratio_latency_artifacts = {} #비율별 모델별 지연시간 측정 결과를 저장할 딕셔너리
ratio_split_rows = [] # 각 비율의 train/valid/test 분할 정보를 저장할 리스트

for ratio_index, config in enumerate(RATIO_CONFIGS): #비율별 반복 시작
    ratio_name = config['ratio_name']
    ratio_label = config['ratio_label']
    ratio_df = ratio_sample_frames[ratio_name] #현재 비율에 해당하는 df 가져오기
    ratio_run_root = RUN_ROOT / f"ratio_{ratio_name}"
    ratio_run_root.mkdir(parents=True, exist_ok=True) #폴더 생성

    #현재 실행 중인 비율, 전체 행수, 라벨 정보 출력
    print('=' * 100)
    print(f'Ratio experiment: {ratio_label} ({ratio_name})')
    print('Dataset rows:', len(ratio_df))
    print('Label counts:', ratio_df['label'].value_counts().sort_index().to_dict())

    #그룹 스플릿으로 train/valid/test 분할
    train_df, valid_df, test_df, split_seed = group_train_valid_test_split(
        ratio_df,
        group_col=SPLIT_GROUP_COL,
        random_seed=RANDOM_SEED + ratio_index * 100, #비율별로 서로 다른 seed를 줌
        max_attempts=100,
    )

    #split 결과 출력
    print('Split seed used    :', split_seed)
    print('Split group column:', SPLIT_GROUP_COL)
    print_split_summary('train', train_df)
    print_split_summary('valid', valid_df)
    print_split_summary('test', test_df)

    #split 요약 정보 저장
    for split_name, split_df in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
        split_counts = split_df['label'].value_counts().sort_index().to_dict()
        ratio_split_rows.append(
            {
                'ratio_name': ratio_name,
                'ratio_label': ratio_label,
                'split': split_name,
                'rows': int(len(split_df)),
                'normal_count': int(split_counts.get(0, 0)),
                'negative_count': int(split_counts.get(1, 0)),
                'positive_rate': float(split_counts.get(1, 0) / len(split_df)),
                'unique_source_prompt_hash': int(split_df[SPLIT_GROUP_COL].nunique()),
                'split_seed': int(split_seed),
            }
        )

    #현재 비율의 결과 저장 공간 만들기
    ratio_experiment_results[ratio_name] = {}
    ratio_latency_artifacts[ratio_name] = {}

    # 모델별 반복 실행
    for model_name in get_model_order_for_ratio(ratio_name):
        print('-' * 100)
        print(f'Running model: {model_name} | ratio={ratio_label}')

        if model_name == 'cosine_similarity':
            result = run_cosine_similarity_experiment( #코사인 유사도 실험
                train_df=train_df,
                valid_df=valid_df,
                test_df=test_df,
                run_root=ratio_run_root,
            )
        elif model_name == 'isolation_forest':
            result = run_isolation_forest_experiment( #Isolation Forest 실험
                train_df=train_df,
                valid_df=valid_df,
                test_df=test_df,
                feature_cols=feature_cols,
                run_root=ratio_run_root,
            )
        else:
            result = run_supervised_experiment( #지도학습 모델
                model_name=model_name,
                train_df=train_df,
                valid_df=valid_df,
                test_df=test_df,
                feature_cols=feature_cols,
                run_root=ratio_run_root,
            )

        #성능 출력
        ratio_experiment_results[ratio_name][model_name] = result
        print('Best threshold :', result['summary']['best_validation_threshold'])
        print('Precision      :', result['summary']['precision'])
        print('Recall         :', result['summary']['recall'])
        print('F1             :', result['summary']['f1'])
        print('Accuracy       :', result['summary']['accuracy'])
        print('AUROC          :', result['summary']['auroc'])
        print('AUPRC          :', result['summary']['auprc'])
        print('Model device   :', result['summary']['model_device_type'])
        print('Output dir     :', result['summary']['output_dir'])

        #latency benchmark 실행
        if RUN_LATENCY_BENCHMARK:
            latency_samples_df, latency_summary, latency_report_df = benchmark_latency(result)
            ratio_latency_artifacts[ratio_name][model_name] = {
                'samples': latency_samples_df,
                'summary': latency_summary,
                'report': latency_report_df,
            }
            print('Latency model inference mean (ms):', latency_summary['model_inference_ms_mean'])

#split summary 저장
ratio_split_summary_df = pd.DataFrame(ratio_split_rows)
ratio_split_summary_path = RUN_ROOT / 'ratio_split_summary.csv'
ratio_split_summary_df.to_csv(ratio_split_summary_path, index=False, encoding='utf-8-sig')

print('Ratio split summary path:', ratio_split_summary_path)
display(ratio_split_summary_df)


## Cell 11. Aggregate ratio-study tables


In [ ]:
#결과 저장용 리스트 생성
metric_rows = []
latency_rows = [] 

sampling_info = ratio_sampling_plan_df.set_index('ratio_name').to_dict(orient='index') #비율별 샘플링 정보를 딕셔너리로 변환
split_info = (
    ratio_split_summary_df[ratio_split_summary_df['split'] == 'test'] #test split 정보만 가져옴
    .set_index('ratio_name')
    .to_dict(orient='index')
)

#비율별 모델별 성능 결과 수집
for config in RATIO_CONFIGS:
    ratio_name = config['ratio_name']
    ratio_label = config['ratio_label']
    for model_name in get_model_order_for_ratio(ratio_name):
        result = ratio_experiment_results[ratio_name][model_name]
        summary = result['summary']
        sample_summary = sampling_info[ratio_name]
        test_split_summary = split_info[ratio_name]

        #성능 지표 행 추가
        metric_rows.append(
            {
                'ratio_name': ratio_name,
                'ratio_label': ratio_label,
                'model_name': model_name,
                'normal_count': int(sample_summary['normal_count']),
                'negative_count': int(sample_summary['negative_count']),
                'dataset_positive_rate': float(sample_summary['positive_rate']),
                'test_normal_count': int(test_split_summary['normal_count']),
                'test_negative_count': int(test_split_summary['negative_count']),
                'test_positive_rate': float(test_split_summary['positive_rate']),
                'score_kind': summary['score_kind'],
                'best_validation_threshold': summary['best_validation_threshold'],
                'precision': summary['precision'],
                'recall': summary['recall'],
                'f1': summary['f1'],
                'accuracy': summary['accuracy'],
                'auroc': summary['auroc'],
                'auprc': summary['auprc'],
                'tn': summary['tn'],
                'fp': summary['fp'],
                'fn': summary['fn'],
                'tp': summary['tp'],
                'fit_seconds': summary['fit_seconds'],
                'embedding_device': summary['embedding_device'],
                'model_device_type': summary['model_device_type'],
                'output_dir': summary['output_dir'],
            }
        )

        #latency 결과 수집
        if RUN_LATENCY_BENCHMARK and model_name in ratio_latency_artifacts[ratio_name]:
            latency_summary = ratio_latency_artifacts[ratio_name][model_name]['summary']
            latency_rows.append(
                {
                    'ratio_name': ratio_name,
                    'ratio_label': ratio_label,
                    'model_name': model_name,
                    'embedding_ms_mean': latency_summary['embedding_ms_mean'],
                    'embedding_ms_median': latency_summary['embedding_ms_median'],
                    'embedding_ms_p95': latency_summary['embedding_ms_p95'],
                    'model_inference_ms_mean': latency_summary['model_inference_ms_mean'],
                    'model_inference_ms_median': latency_summary['model_inference_ms_median'],
                    'model_inference_ms_p95': latency_summary['model_inference_ms_p95'],
                    'end_to_end_ms_mean': latency_summary['end_to_end_ms_mean'],
                    'end_to_end_ms_median': latency_summary['end_to_end_ms_median'],
                    'end_to_end_ms_p95': latency_summary['end_to_end_ms_p95'],
                }
            )

#DataFrame으로 변환
ratio_metrics_df = pd.DataFrame(metric_rows)
ratio_latency_df = pd.DataFrame(latency_rows)

#비율과 모델 순서 지정
ratio_order = [cfg['ratio_name'] for cfg in RATIO_CONFIGS]
ratio_label_order = [cfg['ratio_label'] for cfg in RATIO_CONFIGS]
ratio_metrics_df['ratio_name'] = pd.Categorical(ratio_metrics_df['ratio_name'], categories=ratio_order, ordered=True) #ratio_name을 단순 문자열이 아니라 정해진 순서를 가진 categorical 값으로 변환
ratio_metrics_df['model_name'] = pd.Categorical(ratio_metrics_df['model_name'], categories=MODEL_ORDER, ordered=True) #모델 순서 고정
if not ratio_latency_df.empty:
    ratio_latency_df['ratio_name'] = pd.Categorical(ratio_latency_df['ratio_name'], categories=ratio_order, ordered=True) 
    ratio_latency_df['model_name'] = pd.Categorical(ratio_latency_df['model_name'], categories=MODEL_ORDER, ordered=True)

#정렬
ratio_metrics_df = ratio_metrics_df.sort_values(['ratio_name', 'model_name']).reset_index(drop=True)
ratio_latency_df = ratio_latency_df.sort_values(['ratio_name', 'model_name']).reset_index(drop=True)

#CSV 저장
ratio_metrics_path = RUN_ROOT / 'ratio_metrics_comparison.csv'
ratio_latency_path = RUN_ROOT / 'ratio_latency_comparison.csv'
ratio_metrics_df.to_csv(ratio_metrics_path, index=False, encoding='utf-8-sig')
ratio_latency_df.to_csv(ratio_latency_path, index=False, encoding='utf-8-sig')

print('Ratio metrics path :', ratio_metrics_path)
print('Ratio latency path :', ratio_latency_path)
display(ratio_metrics_df) #전체 성능표 출력

#피벗 테이블 출력
print('AUPRC by ratio and model:')
display(
    ratio_metrics_df
    .pivot(index='ratio_label', columns='model_name', values='auprc')
    .reindex(ratio_label_order)
)

print('AUROC by ratio and model:')
display(
    ratio_metrics_df
    .pivot(index='ratio_label', columns='model_name', values='auroc')
    .reindex(ratio_label_order)
)

print('F1 by ratio and model:')
display(
    ratio_metrics_df
    .pivot(index='ratio_label', columns='model_name', values='f1')
    .reindex(ratio_label_order)
)


## Cell 12. Plot ratio-study curves


In [ ]:
#결과 그래프 출력과 저장
ratio_plot_path = RUN_ROOT / 'ratio_study_metric_curves.png' 
metrics_to_plot = [
    ('auroc', 'AUROC'),
    ('auprc', 'AUPRC'),
    ('f1', 'F1-score'),
    ('recall', 'Recall'),
]

plot_model_order = [
    'lightgbm',
    'xgboost',
    'random_forest',
    'isolation_forest',
]

plot_model_colors = {
    'lightgbm': '#1f77b4',
    'xgboost': '#ff7f0e',
    'random_forest': '#2ca02c',
    'isolation_forest': '#d62728',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.ravel()
x = np.arange(len(ratio_label_order))

for ax, (metric_name, metric_label) in zip(axes, metrics_to_plot):
    for model_name in plot_model_order:
        subset = (
            ratio_metrics_df[ratio_metrics_df['model_name'] == model_name]
            .set_index('ratio_label')
            .reindex(ratio_label_order)
        )
        ax.plot(
            x,
            subset[metric_name].to_numpy(dtype=float),
            marker='o',
            linewidth=2,
            color=plot_model_colors[model_name],
            label=model_name,
        )

    ax.set_title(metric_label)
    ax.set_xticks(x)
    ax.set_xticklabels(ratio_label_order)
    ax.set_xlabel('Normal:Mismatch Ratio')
    ax.set_ylabel(metric_label)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(ratio_plot_path, dpi=200, bbox_inches='tight')
plt.show()

positive_rate_plot_path = RUN_ROOT / 'ratio_study_positive_rate_baseline.png'
fig, ax = plt.subplots(figsize=(8, 5))
baseline_df = (
    ratio_metrics_df[['ratio_label', 'dataset_positive_rate', 'test_positive_rate']]
    .drop_duplicates()
    .set_index('ratio_label')
    .reindex(ratio_label_order)
)
ax.plot(x, baseline_df['dataset_positive_rate'], marker='o', label='dataset positive rate')
ax.plot(x, baseline_df['test_positive_rate'], marker='s', label='test positive rate')
ax.set_xticks(x)
ax.set_xticklabels(ratio_label_order)
ax.set_xlabel('Normal:Mismatch Ratio')
ax.set_ylabel('positive rate')
ax.set_title('AUPRC baseline changes with class prevalence')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(positive_rate_plot_path, dpi=200, bbox_inches='tight')
plt.show()

print('Ratio metric plot path       :', ratio_plot_path)
print('Positive-rate baseline path  :', positive_rate_plot_path)


## Cell 13. Output paths


In [ ]:
print('Run root:', RUN_ROOT)
print('Embedding device (used to build delta features):', EMBEDDING_DEVICE)
print('Model devices: LightGBM=', LIGHTGBM_DEVICE_CANDIDATES, ', XGBoost=', XGBOOST_DEVICE_CANDIDATES)
print()

print('Main outputs:')
print('  sampling plan:', RUN_ROOT / 'ratio_sampling_plan.csv')
print('  split summary:', RUN_ROOT / 'ratio_split_summary.csv')
print('  metrics      :', RUN_ROOT / 'ratio_metrics_comparison.csv')
print('  latency      :', RUN_ROOT / 'ratio_latency_comparison.csv')
print('  metric plot  :', RUN_ROOT / 'ratio_study_metric_curves.png')
print()

for config in RATIO_CONFIGS:
    ratio_name = config['ratio_name']
    ratio_label = config['ratio_label']
    print(f'[{ratio_label}] ratio_{ratio_name}')
    print('  sampled dataset:', RUN_ROOT / f"sampled_dataset_ratio_{ratio_name}.csv")
    for model_name in get_model_order_for_ratio(ratio_name):
        result = ratio_experiment_results[ratio_name][model_name]
        print(f'  {model_name}:', result['summary']['output_dir'])
    print()


## Cell 14. Export mismatch-labeled individual figures

This cell reuses the completed experiment outputs and saves paper-ready figures using Normal/Mismatch terminology.


In [ ]:
#개별 그림 저장장

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


INDIVIDUAL_FIG_DIR = RUN_ROOT / "paper_mismatch_individual_figures"
INDIVIDUAL_FIG_DIR.mkdir(parents=True, exist_ok=True)

RATIO_ORDER_MISMATCH = ["1:1", "7:3", "8:2", "10:1", "100:1"]
MODEL_ORDER_MISMATCH = ["lightgbm", "xgboost", "random_forest", "isolation_forest"]
MODEL_LABELS_MISMATCH = {
    "lightgbm": "LightGBM",
    "xgboost": "XGBoost",
    "random_forest": "Random Forest",
    "isolation_forest": "Isolation Forest",
}
MODEL_COLORS_MISMATCH = {
    "lightgbm": "#1f77b4",
    "xgboost": "#ff7f0e",
    "random_forest": "#2ca02c",
    "isolation_forest": "#d62728",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def save_current_figure(stem):
    png_path = INDIVIDUAL_FIG_DIR / f"{stem}.png"
    svg_path = INDIVIDUAL_FIG_DIR / f"{stem}.svg"
    plt.savefig(png_path, bbox_inches="tight")
    plt.savefig(svg_path, bbox_inches="tight")
    print("Saved:", png_path)
    print("Saved:", svg_path)
    return png_path, svg_path


# Load metric results if this cell is run in a fresh kernel after the experiment.
if "ratio_metrics_df" not in globals():
    ratio_metrics_df = pd.read_csv(RUN_ROOT / "ratio_metrics_comparison.csv")

for col in ["precision", "recall", "f1", "accuracy", "auroc", "auprc"]:
    ratio_metrics_df[col] = pd.to_numeric(ratio_metrics_df[col], errors="coerce")

ratio_metrics_df["ratio_label"] = pd.Categorical(
    ratio_metrics_df["ratio_label"],
    categories=RATIO_ORDER_MISMATCH,
    ordered=True,
)

# 1. Save each ratio-study metric curve separately.
metric_specs = [
    ("auroc", "AUROC"),
    ("auprc", "AUPRC"),
    ("f1", "F1-score"),
    ("recall", "Recall"),
]
x_positions = np.arange(len(RATIO_ORDER_MISMATCH))

for metric_col, metric_label in metric_specs:
    fig, ax = plt.subplots(figsize=(7.2, 4.8))

    for model_name in MODEL_ORDER_MISMATCH:
        subset = (
            ratio_metrics_df[ratio_metrics_df["model_name"] == model_name]
            .set_index("ratio_label")
            .reindex(RATIO_ORDER_MISMATCH)
        )
        ax.plot(
            x_positions,
            subset[metric_col].to_numpy(dtype=float),
            marker="o",
            linewidth=2,
            markersize=5,
            color=MODEL_COLORS_MISMATCH[model_name],
            label=MODEL_LABELS_MISMATCH[model_name],
        )

    ax.set_title(metric_label)
    ax.set_xlabel("Normal:Mismatch Ratio")
    ax.set_ylabel(metric_label)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(RATIO_ORDER_MISMATCH)
    ax.set_ylim(0.0, 1.03)
    ax.legend(loc="best")
    fig.tight_layout()

    save_current_figure(f"mismatch_metric_curve_{metric_col}")
    plt.show()


# 2. Delta embedding L2 norm, PCA, and centroid-projection figures.
if "feature_df" not in globals() or "feature_cols" not in globals():
    raise RuntimeError("feature_df and feature_cols are required. Run the delta feature generation cells first.")

ANALYSIS_SAMPLE_PER_LABEL_LOCAL = globals().get("ANALYSIS_SAMPLE_PER_LABEL", 10000)
SILHOUETTE_SAMPLE_PER_LABEL_LOCAL = globals().get("SILHOUETTE_SAMPLE_PER_LABEL", 1000)

analysis_parts = []
for label_value in [0, 1]:
    subset = feature_df[feature_df["label"] == label_value]
    sample_size = min(ANALYSIS_SAMPLE_PER_LABEL_LOCAL, len(subset))
    analysis_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + label_value))

analysis_df = (
    pd.concat(analysis_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

analysis_delta_matrix = analysis_df[feature_cols].to_numpy(dtype=np.float32)
analysis_labels = analysis_df["label"].to_numpy(dtype=int)
analysis_delta_norms = np.linalg.norm(analysis_delta_matrix, axis=1).astype(np.float32)

plot_df = analysis_df[["label"]].copy()
plot_df["delta_l2_norm"] = analysis_delta_norms

# 2-1. L2 norm distribution.
fig, ax = plt.subplots(figsize=(7.2, 4.8))
for label_value, color, label_name in [
    (0, "#1f77b4", "Normal (label 0)"),
    (1, "#d62728", "Mismatch (label 1)"),
]:
    subset = plot_df[plot_df["label"] == label_value]
    ax.hist(
        subset["delta_l2_norm"],
        bins=40,
        alpha=0.55,
        density=True,
        color=color,
        label=label_name,
    )

ax.set_title("Delta Embedding L2 Norm Distribution")
ax.set_xlabel("L2 norm of prompt_emb - tool_emb")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
save_current_figure("mismatch_delta_l2_norm_histogram")
plt.show()

l2_summary_df = (
    plot_df.groupby("label")["delta_l2_norm"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)
l2_quantile_df = (
    plot_df.groupby("label")["delta_l2_norm"]
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    .unstack()
    .reset_index()
)
l2_summary_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_l2_norm_summary.csv"
l2_quantile_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_l2_norm_quantiles.csv"
l2_summary_df.to_csv(l2_summary_path, index=False, encoding="utf-8-sig")
l2_quantile_df.to_csv(l2_quantile_path, index=False, encoding="utf-8-sig")
print("Saved:", l2_summary_path)
print("Saved:", l2_quantile_path)
display(l2_summary_df)
display(l2_quantile_df)

# 2-2. PCA scatter.
pca = PCA(n_components=2, random_state=RANDOM_SEED)
delta_pca_2d = pca.fit_transform(analysis_delta_matrix)

pca_df = plot_df.copy()
pca_df["pca_x"] = delta_pca_2d[:, 0]
pca_df["pca_y"] = delta_pca_2d[:, 1]

fig, ax = plt.subplots(figsize=(7.2, 5.4))
for label_value, color, label_name in [
    (0, "#1f77b4", "Normal (label 0)"),
    (1, "#d62728", "Mismatch (label 1)"),
]:
    subset = pca_df[pca_df["label"] == label_value]
    ax.scatter(
        subset["pca_x"],
        subset["pca_y"],
        s=12,
        alpha=0.55,
        color=color,
        label=label_name,
    )

ax.set_title("Delta Embedding PCA")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.legend()
fig.tight_layout()
save_current_figure("mismatch_delta_pca_scatter")
plt.show()

pca_csv_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_pca_sample_coordinates.csv"
pca_summary_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_pca_summary.csv"
pca_df.to_csv(pca_csv_path, index=False, encoding="utf-8-sig")
pca_summary_df = pd.DataFrame(
    {
        "component": ["PCA1", "PCA2"],
        "explained_variance_ratio": pca.explained_variance_ratio_,
    }
)
pca_summary_df.to_csv(pca_summary_path, index=False, encoding="utf-8-sig")
print("Saved:", pca_csv_path)
print("Saved:", pca_summary_path)
display(pca_summary_df)

# 2-3. Projection on centroid-difference axis.
normal_matrix = analysis_delta_matrix[analysis_labels == 0]
mismatch_matrix = analysis_delta_matrix[analysis_labels == 1]

normal_centroid = normal_matrix.mean(axis=0)
mismatch_centroid = mismatch_matrix.mean(axis=0)
centroid_delta = mismatch_centroid - normal_centroid
centroid_delta_norm = float(np.linalg.norm(centroid_delta))

normal_centroid_norm = float(np.linalg.norm(normal_centroid))
mismatch_centroid_norm = float(np.linalg.norm(mismatch_centroid))
centroid_cosine = float(
    np.dot(normal_centroid, mismatch_centroid)
    / max(normal_centroid_norm * mismatch_centroid_norm, 1e-12)
)

normal_spread = np.linalg.norm(normal_matrix - normal_centroid, axis=1)
mismatch_spread = np.linalg.norm(mismatch_matrix - mismatch_centroid, axis=1)

if centroid_delta_norm > 0:
    separation_axis = centroid_delta / centroid_delta_norm
else:
    separation_axis = np.zeros_like(centroid_delta)

projection_scores = analysis_delta_matrix @ separation_axis
projection_df = pd.DataFrame(
    {
        "label": analysis_labels,
        "projection_score": projection_scores,
    }
)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
for label_value, color, label_name in [
    (0, "#1f77b4", "Normal (label 0)"),
    (1, "#d62728", "Mismatch (label 1)"),
]:
    subset = projection_df[projection_df["label"] == label_value]
    ax.hist(
        subset["projection_score"],
        bins=40,
        alpha=0.55,
        density=True,
        color=color,
        label=label_name,
    )

ax.set_title("Projection on Centroid-Difference Axis")
ax.set_xlabel("Projection score")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
save_current_figure("mismatch_delta_centroid_projection_histogram")
plt.show()

projection_csv_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_centroid_projection_scores.csv"
projection_quantile_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_centroid_projection_quantiles.csv"
separation_summary_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_separation_summary.csv"
projection_df.to_csv(projection_csv_path, index=False, encoding="utf-8-sig")

projection_quantile_df = (
    projection_df.groupby("label")["projection_score"]
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    .unstack()
    .reset_index()
)
projection_quantile_df.to_csv(projection_quantile_path, index=False, encoding="utf-8-sig")

silhouette_parts = []
for label_value in [0, 1]:
    subset = analysis_df[analysis_df["label"] == label_value]
    sample_size = min(SILHOUETTE_SAMPLE_PER_LABEL_LOCAL, len(subset))
    silhouette_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + 100 + label_value))

silhouette_df = pd.concat(silhouette_parts, ignore_index=True)
silhouette_matrix = silhouette_df[feature_cols].to_numpy(dtype=np.float32)
silhouette_labels = silhouette_df["label"].to_numpy(dtype=int)

try:
    silhouette = float(silhouette_score(silhouette_matrix, silhouette_labels, metric="euclidean"))
except Exception as exc:
    silhouette = None
    print("Silhouette score could not be computed:", exc)

separation_summary_df = pd.DataFrame(
    [
        {"metric": "centroid_l2_distance", "value": centroid_delta_norm},
        {"metric": "centroid_cosine_similarity", "value": centroid_cosine},
        {"metric": "normal_within_class_mean_l2", "value": float(normal_spread.mean())},
        {"metric": "mismatch_within_class_mean_l2", "value": float(mismatch_spread.mean())},
        {
            "metric": "normal_projection_mean",
            "value": float(projection_df.loc[projection_df["label"] == 0, "projection_score"].mean()),
        },
        {
            "metric": "mismatch_projection_mean",
            "value": float(projection_df.loc[projection_df["label"] == 1, "projection_score"].mean()),
        },
        {"metric": "silhouette_score_sampled", "value": silhouette},
    ]
)
separation_summary_df.to_csv(separation_summary_path, index=False, encoding="utf-8-sig")
print("Saved:", projection_csv_path)
print("Saved:", projection_quantile_path)
print("Saved:", separation_summary_path)
display(projection_quantile_df)
display(separation_summary_df)

print()
print("All mismatch-labeled outputs saved to:", INDIVIDUAL_FIG_DIR)
